In [1]:
import os
import pandas as pd
from datetime import datetime
import json
import gc

folder_path_demanddetails = '/home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/'

# read active properties & needed columns
property_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/eg_pt_property.csv',
    usecols=['id', 'propertyid', 'tenantid', 'createdtime', 'additionaldetails', 'ownershipcategory', 'status', 'usagecategory']
)
property_df = property_df[property_df['status'] == 'ACTIVE'].copy()

# read units
# unit_df = pd.read_csv(
#     '/home/prerna/Punjab/punjab-data-prod-analysis/srihargobindpur/eg_pt_unit.csv',
#     usecols=['propertyid', 'occupancytype']
# )



# read demand
demand_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/egbs_demand_v1.csv',
    dtype={"consumercode": str},
    low_memory=False,
    usecols=['id', 'taxperiodfrom', 'taxperiodto', 'consumercode', 'status', 'businessservice']
)
demand_df = demand_df[demand_df['status'] == 'ACTIVE'].copy()
demand_df = demand_df[demand_df['businessservice'] == 'PT'].copy()


# read demand details (memory‑efficient, in chunks)
all_chunks = []
needed_cols = ['demandid', 'taxamount', 'collectionamount', 'taxheadcode']
for filename in os.listdir(folder_path_demanddetails):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path_demanddetails, filename)
        print(f'Loading: {file_path}')
        chunk = pd.read_csv(file_path, usecols=needed_cols)
        all_chunks.append(chunk)
demand_details_df = pd.concat(all_chunks, ignore_index=True)
del all_chunks; gc.collect()

print("✅ Loaded data")

Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/output_263.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/output_4.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/output_753.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/output_479.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/output_555.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/output_672.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/output_496.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/output_718.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_details/output_545.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/jalandhar/output_demand_detai

In [2]:
print(len(property_df))         # number of rows in properties
# print(len(unit_df))             # number of rows in units
print(len(demand_df))   # number of rows in demand details
print(len(demand_details_df))   # number of rows in demand details

184070
1199777
39269965


In [3]:
# join demand and demand details
joined_demand = demand_df.merge(demand_details_df, left_on='id', right_on='demandid', how='left', suffixes=('_demand', '_detail'))
print(joined_demand['id'].nunique())
del demand_details_df, demand_df; gc.collect()
joined_demand.head()

1199777


,id,consumercode,businessservice,taxperiodfrom,taxperiodto,status,demandid,taxheadcode,taxamount,collectionamount
0,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT-1013-467682,PT,1364774400000,1396310399000,ACTIVE,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT_FIRE_CESS,0.00,0.00
1,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT-1013-467682,PT,1364774400000,1396310399000,ACTIVE,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT_ROUNDOFF,0.16,0.16
2,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT-1013-467682,PT,1364774400000,1396310399000,ACTIVE,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT_OWNER_EXEMPTION,0.00,0.00
3,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT-1013-467682,PT,1364774400000,1396310399000,ACTIVE,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT_TIME_PENALTY,0.00,0.00
4,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT-1013-467682,PT,1364774400000,1396310399000,ACTIVE,fcb1e7b4-0588-4acb-9840-ca3d77612b8c,PT_TIME_INTEREST,0.00,0.00


In [4]:
import pytz

# Correct: parse as datetime from milliseconds since epoch
joined_demand['taxperiodfrom'] = pd.to_datetime(joined_demand['taxperiodfrom'], unit='ms', utc=True)
joined_demand['taxperiodto'] = pd.to_datetime(joined_demand['taxperiodto'], unit='ms', utc=True)

# Convert to IST (Asia/Kolkata)
ist = pytz.timezone('Asia/Kolkata')
joined_demand['taxperiodfrom'] = joined_demand['taxperiodfrom'].dt.tz_convert(ist)
joined_demand['taxperiodto'] = joined_demand['taxperiodto'].dt.tz_convert(ist)

# Financial year calculation
def get_fy(date):
    if date.month >= 4:
        fy_start = date.year
        fy_end = date.year + 1
    else:
        fy_start = date.year - 1
        fy_end = date.year
    return f"{fy_start}-{str(fy_end)[-2:]}"

joined_demand['fy'] = joined_demand['taxperiodfrom'].apply(get_fy)

# Group by consumercode
result = joined_demand.groupby('consumercode')['fy'].agg(['min', 'max']).reset_index()
result.rename(columns={'min': 'earliest_fy', 'max': 'latest_fy'}, inplace=True)

print(result)

           consumercode earliest_fy latest_fy
0       PT-1013-1000009     2013-14   2024-25
1       PT-1013-1000020     2018-19   2024-25
2       PT-1013-1000024     2020-21   2024-25
3       PT-1013-1000029     2020-21   2024-25
4       PT-1013-1000037     2020-21   2024-25
...                 ...         ...       ...
187894   PT-1013-999879     2013-14   2024-25
187895   PT-1013-999958     2020-21   2024-25
187896   PT-1013-999960     2020-21   2024-25
187897   PT-1013-999961     2020-21   2024-25
187898   PT-1013-999975     2014-15   2024-25

[187899 rows x 3 columns]


In [5]:
# Merge latest_fy onto joined_demand by consumercode
joined = joined_demand.merge(
    result[['consumercode', 'latest_fy']],
    on='consumercode',
    how='left'
)

# Filter only latest FY
latest_demand = joined[joined['fy'] == joined['latest_fy']]

# Pivot taxheadcode values into separate columns
pivoted = latest_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
# PT_TAX + PT_CANCER_CESS + PT_FIRE_CESS + PT_ROUNDOFF - (PT_OWNER_EXEMPTION + PT_UNIT_USAGE_EXEMPTION)
pivoted['latest_fy_taxamount'] = (
    pivoted.get('PT_TAX', 0) +
    pivoted.get('PT_CANCER_CESS', 0) +
    pivoted.get('PT_FIRE_CESS', 0) +
    pivoted.get('PT_ROUNDOFF', 0) -
    ( pivoted.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Merge back into result
result = result.merge(
    pivoted[['consumercode', 'latest_fy_taxamount']],
    on='consumercode',
    how='left'
)

print(result.head())


      consumercode earliest_fy latest_fy  latest_fy_taxamount
0  PT-1013-1000009     2013-14   2024-25                 0.00
1  PT-1013-1000020     2018-19   2024-25              5530.92
2  PT-1013-1000024     2020-21   2024-25              5778.00
3  PT-1013-1000029     2020-21   2024-25              8077.00
4  PT-1013-1000037     2020-21   2024-25               772.74


In [6]:
# Calculating the tax amount (demand) of current year using formula
target_fy = "2025-26"
current_fy_demand = joined_demand[joined_demand['fy'] == target_fy]

# Pivot taxheadcode values into separate columns
pivoted_current = current_fy_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
pivoted_current['current_fy_taxamount'] = (
    pivoted_current.get('PT_TAX', 0) +
    pivoted_current.get('PT_CANCER_CESS', 0) +
    pivoted_current.get('PT_FIRE_CESS', 0) +
    pivoted_current.get('PT_ROUNDOFF', 0) -
    ( pivoted_current.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted_current.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Keep only required cols
pivoted_current = pivoted_current[['consumercode', 'current_fy_taxamount']]

# Ensure all consumercodes are present
all_consumercodes = pd.DataFrame(joined_demand['consumercode'].unique(), columns=['consumercode'])
final = all_consumercodes.merge(pivoted_current, on='consumercode', how='left')
final['current_fy_taxamount'] = final['current_fy_taxamount'].fillna(0)

# Merge into result
result = result.merge(final, on='consumercode', how='left')
result['current_fy_taxamount'] = result['current_fy_taxamount'].fillna(0)

print(result.head())


      consumercode earliest_fy latest_fy  latest_fy_taxamount  \
0  PT-1013-1000009     2013-14   2024-25                 0.00   
1  PT-1013-1000020     2018-19   2024-25              5530.92   
2  PT-1013-1000024     2020-21   2024-25              5778.00   
3  PT-1013-1000029     2020-21   2024-25              8077.00   
4  PT-1013-1000037     2020-21   2024-25               772.74   

   current_fy_taxamount  
0                   0.0  
1                   0.0  
2                   0.0  
3                   0.0  
4                   0.0  


In [7]:
property_result_merged = property_df.merge(
    result,
    left_on='propertyid',
    right_on='consumercode',
    how='left'
)

print(property_result_merged)

                                          id       propertyid      tenantid  \
0       602a42b8-1f2a-40d0-8cd3-86f4b29fad56   PT-1013-460425  pb.jalandhar   
1       6358083c-6576-4fa7-ad1e-f8418d1b62dc  PT-1013-2352280  pb.jalandhar   
2       14fabe93-2049-4689-9ca4-c0bc5a6d2f55  PT-1013-1151516  pb.jalandhar   
3       db8bcfb8-fa0a-4f42-aec4-77e4bf1d6619  PT-1013-1446443  pb.jalandhar   
4       f44e91ab-60c4-40fd-86c2-16fcced74c26   PT-1013-914284  pb.jalandhar   
...                                      ...              ...           ...   
184065  5ca379a7-e05b-4899-9276-d29a0c647338   PT-1013-540818  pb.jalandhar   
184066  3cddb513-9c7c-4ba8-bca5-f0cdaf2a942e   PT-1013-540806  pb.jalandhar   
184067  442e74c5-a5b2-443e-86a1-547722470459   PT-1013-707131  pb.jalandhar   
184068  d897a493-83a3-4db3-b322-0785ad2d3df2   PT-1013-987939  pb.jalandhar   
184069  8e520ed2-8bb1-4dc2-bfc4-a0d5b84b8ba8  PT-1013-1774030  pb.jalandhar   

        status          ownershipcategory          

In [8]:
property_result_merged.to_csv('Punjab_Data_Analysis_jalandhar_final_2.csv', index=False)